# Week 6 — Isolation Forest Anomaly Detection, SR-04 Threshold Calibration & Unseen-Attack Holdout

**Topic 04 (NT219 / NT2205):** *Agentic ML/DL for Metadata-Only TLS/SSH Threat Detection, Hunting & Response*

### Design & Compliance Checklist
- **Benign-Only Baseline Fit**: Isolation Forest is trained strictly on `label == 0` flows from the Train split.
- **SR-04 Validation Calibration**: Calibrates decision threshold on the Validation split at fixed benign FPR (`target_fpr = 0.05`) and emits calibrated confidence + contextual `reason_codes`.
- **Unseen-Attack Evaluation (Section 14)**: Evaluates on `test/unseen_attack_holdout.parquet` (`Shellcode`, `Worms`) paired with benign Test flows, reporting `calibrated_threshold`, `aucpr`, `roc_auc`, and detection rate (`tpr_at_fpr`).

## Cell 1 — Configuration, Path Resolution & Reproducibility

In [1]:
import json
import math
import os
import platform
import random
import re
import sys
import time
import warnings
from collections.abc import Mapping
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, FrozenSet, List, Optional, Sequence, Tuple

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch

warnings.filterwarnings("ignore", category=UserWarning)

L3_L4_FEATURE_COLUMNS: List[str] = [
    "src_port_class", "dst_port", "proto", "in_bytes", "in_pkts", "out_bytes", "out_pkts",
    "tcp_flags", "client_tcp_flags", "server_tcp_flags", "flow_duration_ms", "duration_in",
    "duration_out", "min_ttl", "max_ttl", "longest_flow_pkt", "shortest_flow_pkt",
    "min_ip_pkt_len", "max_ip_pkt_len", "src_to_dst_second_bytes", "dst_to_src_second_bytes",
    "retransmitted_in_bytes", "retransmitted_in_pkts", "retransmitted_out_bytes",
    "retransmitted_out_pkts", "src_to_dst_avg_throughput", "dst_to_src_avg_throughput",
    "num_pkts_up_to_128_bytes", "num_pkts_128_to_256_bytes", "num_pkts_256_to_512_bytes",
    "num_pkts_512_to_1024_bytes", "num_pkts_1024_to_1514_bytes", "tcp_win_max_in",
    "tcp_win_max_out", "icmp_type", "icmp_ipv4_type", "is_tls_port", "is_ssh_port",
    "bytes_ratio", "pkts_ratio",
]

KAGGLE_DATASET_DIR = Path("/kaggle/input/datasets/vxdwkhoa/Dataset_NF/NF-UQ-NIDS-v2")
KAGGLE_CHECKPOINT_DIR = Path("/kaggle/working/Checkpoint")
LOCAL_DATASET_DIR = Path(r"D:\Crypto\DoAn\NF-UQ-NIDS-v2")
LOCAL_CHECKPOINT_DIR = Path(r"D:\Crypto\DoAn\Notebook\Checkpoint")
REPO_SAMPLE_PATH = Path(r"D:\Crypto\DoAn\topic04-agentic-tls-ssh-threat-detection\data\public_loaders\sample_data\nfuq_sample_1500.parquet")


@dataclass
class ExperimentConfig:
    seed: int = 42
    max_train_rows: Optional[int] = None  # Set e.g. 50000 for fast local tests; None for full Kaggle GPU
    dataset_dir: Optional[str] = None
    checkpoint_dir: Optional[str] = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


class PathResolver:
    """Dual-environment resolver: Explicit/ENV -> Kaggle GPU (+ slug scan) -> Local Full -> Repo Sample."""

    @staticmethod
    def _scan_kaggle_input(kaggle_root: Path = Path("/kaggle/input"), max_depth: int = 4) -> Optional[Path]:
        if not kaggle_root.is_dir():
            return None
        queue: List[Tuple[Path, int]] = [(kaggle_root, 0)]
        while queue:
            current, depth = queue.pop(0)
            if (current / "train" / "flows_tabular.parquet").is_file():
                return current
            if depth < max_depth:
                try:
                    children = sorted((p for p in current.iterdir() if p.is_dir()), key=lambda p: p.name)
                except OSError:
                    children = []
                for child in children:
                    queue.append((child, depth + 1))
        return None

    @classmethod
    def resolve(cls, config: ExperimentConfig) -> Dict[str, Any]:
        env_data = os.environ.get("TOPIC04_DATA_DIR", "").strip()
        env_ckpt = os.environ.get("TOPIC04_CHECKPOINT_DIR", "").strip()

        resolved_data: Optional[Path] = None
        mode = "unknown"
        is_sample = False

        # Resolve dataset path
        candidates: List[Tuple[str, Path]] = []
        if config.dataset_dir:
            candidates.append(("custom", Path(config.dataset_dir)))
        elif env_data:
            candidates.append(("custom", Path(env_data)))
        else:
            candidates.append(("kaggle", KAGGLE_DATASET_DIR))

        def _inspect(cand: Path) -> Optional[Tuple[Path, bool]]:
            if not cand.exists():
                return None
            if cand.is_file() and cand.suffix.lower() == ".parquet":
                return cand, True
            if cand.is_dir():
                if (cand / "train" / "flows_tabular.parquet").is_file():
                    return cand, False
                if (cand / "NF-UQ-NIDS-v2" / "train" / "flows_tabular.parquet").is_file():
                    return cand / "NF-UQ-NIDS-v2", False
                if (cand / "nfuq_sample_1500.parquet").is_file():
                    return cand / "nfuq_sample_1500.parquet", True
            return None

        for label, cand in candidates:
            hit = _inspect(cand)
            if hit is not None:
                resolved_data, is_sample = hit
                mode = label
                break

        if resolved_data is None and not config.dataset_dir and not env_data:
            kaggle_input_root = Path("/kaggle/input")
            if kaggle_input_root.exists():
                slug_dir = cls._scan_kaggle_input(kaggle_input_root, max_depth=4)
                if slug_dir is not None:
                    resolved_data, mode, is_sample = slug_dir, "kaggle_slug_scan", False

        if resolved_data is None and not config.dataset_dir and not env_data:
            for label, cand in (
                ("local_full", LOCAL_DATASET_DIR),
                ("local_sample", REPO_SAMPLE_PATH),
                ("local_sample_rel", Path("data/public_loaders/sample_data/nfuq_sample_1500.parquet")),
                ("local_sample_up", Path("../data/public_loaders/sample_data/nfuq_sample_1500.parquet")),
            ):
                hit = _inspect(cand)
                if hit is not None:
                    resolved_data, is_sample = hit
                    mode = label
                    break

        if resolved_data is None:
            raise FileNotFoundError("Could not locate NF-UQ-NIDS-v2 dataset (Kaggle, local, or sample).")

        # Resolve checkpoint directory
        if config.checkpoint_dir:
            ckpt_dir = Path(config.checkpoint_dir)
        elif env_ckpt:
            ckpt_dir = Path(env_ckpt)
        elif Path("/kaggle/working").exists():
            ckpt_dir = KAGGLE_CHECKPOINT_DIR
        elif LOCAL_CHECKPOINT_DIR.parent.exists():
            ckpt_dir = LOCAL_CHECKPOINT_DIR
        else:
            ckpt_dir = Path.cwd() / "Checkpoint"

        ckpt_dir.mkdir(parents=True, exist_ok=True)
        return {
            "mode": mode,
            "is_sample_mode": is_sample,
            "dataset_path": resolved_data,
            "checkpoint_dir": ckpt_dir,
        }


def set_global_seed(seed: int = 42) -> Dict[str, str]:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    return {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "torch": torch.__version__,
        "joblib": joblib.__version__,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }


CONFIG = ExperimentConfig(seed=42)
VERSIONS = set_global_seed(CONFIG.seed)
PATHS = PathResolver.resolve(CONFIG)
print("Resolved Paths:", {k: str(v) for k, v in PATHS.items()})
print("Reproducibility & Library Versions:", json.dumps({"seed": CONFIG.seed, **VERSIONS}, indent=2))

Resolved Paths: {'mode': 'kaggle_slug_scan', 'is_sample_mode': 'False', 'dataset_path': '/kaggle/input/datasets/vxdwkhoa/dataset-nf/NF-UQ-NIDS-v2', 'checkpoint_dir': '/kaggle/working/Checkpoint'}
Reproducibility & Library Versions: {
  "seed": 42,
  "python": "3.13.18",
  "numpy": "2.1.3",
  "pandas": "2.3.3",
  "scikit-learn": "1.6.1",
  "torch": "2.11.0+cpu",
  "joblib": "1.6.0",
  "device": "cpu"
}


## Cell 2 — Data Loader & No-L7 Privacy Validator

In [2]:
UNSEEN_ATTACK_FAMILIES: FrozenSet[str] = frozenset({"Worms", "Shellcode", "Ransomware"})


class PrivacyAndDataLoader:
    """
    Loads NF-UQ-NIDS-v2 Temporal + Host Holdout + Unseen Attack Holdout splits
    and enforces strict No-L7 Payload privacy validation (SR-01, CR-06, Section 13.4 Step 7).
    """

    FORBIDDEN_PATTERNS = [
        r"^http_", r"^ssh_command", r"^ssh_terminal", r"^ssh_keystroke", r"^ssh_auth_password",
        r"^decrypted_", r"^tls_session_key", r"^private_key", r"^raw_pcap", r"^payload",
        r"^l7_proto$", r"^dns_query_", r"^dns_ttl_", r"^ftp_command_",
        r"^ipv4_src_addr$", r"^ipv4_dst_addr$", r"^raw_s(ni|an)$",
    ]
    IPV4_RE = re.compile(r"^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$")

    def __init__(self, paths: Dict[str, Any]) -> None:
        self.paths = paths

    def validate_dataframe_privacy(self, df: pd.DataFrame, split_name: str) -> Dict[str, Any]:
        violations = []
        for col in df.columns:
            col_l = col.lower()
            for pat in self.FORBIDDEN_PATTERNS:
                if re.search(pat, col_l):
                    violations.append(f"Forbidden L7/raw-IP column detected: {col}")
        for tok_col in ("src_host_token", "dst_host_token"):
            if tok_col in df.columns:
                unique_vals = df[tok_col].dropna().astype(str).unique()
                if any(self.IPV4_RE.match(v) for v in unique_vals):
                    violations.append(f"Raw IPv4 leaked inside {tok_col}")
        if violations:
            raise ValueError(f"Privacy Validator FAILED on {split_name}: {violations}")

        missing_features = [c for c in L3_L4_FEATURE_COLUMNS if c not in df.columns]
        if missing_features:
            raise ValueError(f"Dataset '{split_name}' missing required L3/L4 features: {missing_features}")

        return {"split": split_name, "passed_no_l7_audit": True, "rows": len(df), "violations": 0}

    @staticmethod
    def _sort_by_flow_order(df: pd.DataFrame) -> pd.DataFrame:
        if "flow_order_id" in df.columns and len(df) > 0:
            return df.sort_values("flow_order_id", kind="mergesort").reset_index(drop=True)
        return df.reset_index(drop=True)

    @classmethod
    def _split_sample_temporal(
        cls, df_raw: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        if "attack_family" in df_raw.columns:
            unseen_mask = df_raw["attack_family"].astype(str).isin(UNSEEN_ATTACK_FAMILIES)
        else:
            unseen_mask = pd.Series(False, index=df_raw.index)

        df_unseen = cls._sort_by_flow_order(df_raw.loc[unseen_mask].copy())
        df_known = cls._sort_by_flow_order(df_raw.loc[~unseen_mask].copy())

        n = len(df_known)
        if n == 0:
            df_train = df_known.copy()
            df_val = df_known.iloc[0:0].copy()
            df_test_known = df_known.iloc[0:0].copy()
        elif n == 1:
            df_train = df_known.iloc[:1].reset_index(drop=True)
            df_val = df_known.iloc[0:0].copy()
            df_test_known = df_known.iloc[0:0].copy()
        elif n == 2:
            df_train = df_known.iloc[:1].reset_index(drop=True)
            df_val = df_known.iloc[1:2].reset_index(drop=True)
            df_test_known = df_known.iloc[0:0].copy()
        else:
            n_train = max(1, int(n * 0.70))
            n_val = max(1, int(n * 0.15))
            if n_train + n_val >= n:
                n_train = max(1, n - 2)
                n_val = 1
            df_train = df_known.iloc[:n_train].reset_index(drop=True)
            df_val = df_known.iloc[n_train:n_train + n_val].reset_index(drop=True)
            df_test_known = df_known.iloc[n_train + n_val:].reset_index(drop=True)

        df_test = cls._sort_by_flow_order(
            pd.concat([df_test_known, df_unseen], axis=0, ignore_index=True)
        )
        return df_train, df_val, df_test, df_unseen

    def load_splits(self, max_train_rows: Optional[int] = None) -> Dict[str, pd.DataFrame]:
        dataset_path: Path = self.paths["dataset_path"]
        if self.paths["is_sample_mode"]:
            df_raw = pd.read_parquet(dataset_path)
            self.validate_dataframe_privacy(df_raw, "sample_raw")
            df_train, df_val, df_test, df_unseen = self._split_sample_temporal(df_raw)
        else:
            df_train = pd.read_parquet(dataset_path / "train" / "flows_tabular.parquet")
            df_val = pd.read_parquet(dataset_path / "val" / "flows_tabular.parquet")
            df_test = pd.read_parquet(dataset_path / "test" / "flows_tabular.parquet")
            unseen_path = dataset_path / "test" / "unseen_attack_holdout.parquet"
            if unseen_path.is_file():
                df_unseen = pd.read_parquet(unseen_path)
            elif "attack_family" in df_test.columns:
                df_unseen = df_test[df_test["attack_family"].astype(str).isin(UNSEEN_ATTACK_FAMILIES)].copy()
            else:
                df_unseen = df_test.iloc[0:0].copy()

        if max_train_rows is not None and len(df_train) > max_train_rows:
            df_train = df_train.iloc[:max_train_rows].copy()

        splits = {"train": df_train, "val": df_val, "test": df_test, "unseen_holdout": df_unseen}
        for name, frame in splits.items():
            audit = self.validate_dataframe_privacy(frame, name)
            print(f"[PRIVACY PASS] {audit}")
        return splits


loader = PrivacyAndDataLoader(PATHS)
splits = loader.load_splits(max_train_rows=CONFIG.max_train_rows)

[PRIVACY PASS] {'split': 'train', 'passed_no_l7_audit': True, 'rows': 425771, 'violations': 0}
[PRIVACY PASS] {'split': 'val', 'passed_no_l7_audit': True, 'rows': 91230, 'violations': 0}
[PRIVACY PASS] {'split': 'test', 'passed_no_l7_audit': True, 'rows': 135108, 'violations': 0}
[PRIVACY PASS] {'split': 'unseen_holdout', 'passed_no_l7_audit': True, 'rows': 635, 'violations': 0}


## Cell 3 — Train-Only Feature Sanitization & Scaling

In [3]:
from sklearn.preprocessing import StandardScaler


class FlowFeatureTransformer:
    """
    Train-only feature sanitizer & scaler (SR-08 anti-leakage):
    1. Casts to float64 column-wise and replaces +inf/-inf (empirical overflow in *_second_bytes) with NaN.
    2. Imputes with Train-only finite medians and clips to Train-only quantile bounds.
    3. Fits StandardScaler strictly on Train (clamping scale_ < 1e-12 -> 1.0) and outputs finite float32 arrays.
    """

    def __init__(
        self,
        feature_cols: Sequence[str] = L3_L4_FEATURE_COLUMNS,
        clip_quantiles: Tuple[float, float] = (0.0, 0.9999),
        output_dtype: str = "float32",
    ) -> None:
        if len(feature_cols) == 0:
            raise ValueError("feature_cols must not be empty.")
        if len(clip_quantiles) != 2 or not (0.0 <= clip_quantiles[0] <= clip_quantiles[1] <= 1.0):
            raise ValueError(f"clip_quantiles must satisfy 0.0 <= low <= high <= 1.0, got {clip_quantiles}")
        if output_dtype not in ("float32", "float64"):
            raise ValueError(f"output_dtype must be 'float32' or 'float64', got '{output_dtype}'")

        self.feature_cols: Tuple[str, ...] = tuple(feature_cols)
        self.clip_quantiles: Tuple[float, float] = (float(clip_quantiles[0]), float(clip_quantiles[1]))
        self.output_dtype: str = output_dtype

        self.scaler_: StandardScaler = StandardScaler()
        self.medians_: Optional[np.ndarray] = None
        self.lower_bounds_: Optional[np.ndarray] = None
        self.upper_bounds_: Optional[np.ndarray] = None
        self.is_fitted_: bool = False

    def _validate_columns(self, df: pd.DataFrame) -> None:
        missing = [c for c in self.feature_cols if c not in df.columns]
        if missing:
            raise ValueError(f"DataFrame is missing required feature columns: {missing}")

    def _extract_float64(self, df: pd.DataFrame) -> np.ndarray:
        self._validate_columns(df)
        cols = list(self.feature_cols)
        arrays = []
        for col in cols:
            series = df[col]
            if pd.api.types.is_numeric_dtype(series):
                arr = series.to_numpy(dtype=np.float64, copy=True)
            else:
                arr = pd.to_numeric(series, errors="coerce").to_numpy(dtype=np.float64, copy=True)
            arrays.append(arr)
        X = np.column_stack(arrays).astype(np.float64, copy=False)
        return np.where(np.isfinite(X), X, np.nan)

    def fit(self, df_train: pd.DataFrame) -> "FlowFeatureTransformer":
        if len(df_train) == 0:
            raise ValueError("Cannot fit FlowFeatureTransformer on an empty DataFrame.")

        X = self._extract_float64(df_train)
        n_features = X.shape[1]
        medians = np.zeros(n_features, dtype=np.float64)
        for idx in range(n_features):
            col_vals = X[:, idx]
            finite_mask = np.isfinite(col_vals)
            medians[idx] = float(np.median(col_vals[finite_mask])) if np.any(finite_mask) else 0.0
        self.medians_ = medians

        nan_mask = np.isnan(X)
        if np.any(nan_mask):
            X = np.where(nan_mask, self.medians_, X)

        q_low, q_high = self.clip_quantiles
        self.lower_bounds_ = np.quantile(X, q_low, axis=0).astype(np.float64)
        self.upper_bounds_ = np.quantile(X, q_high, axis=0).astype(np.float64)
        X = np.clip(X, self.lower_bounds_, self.upper_bounds_)

        self.scaler_ = StandardScaler()
        self.scaler_.fit(X)
        self.scaler_.scale_ = np.where(self.scaler_.scale_ < 1e-12, 1.0, self.scaler_.scale_)
        self.is_fitted_ = True
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        if not self.is_fitted_:
            raise RuntimeError("FlowFeatureTransformer must be fitted on Train before transform().")
        self._validate_columns(df)

        out_dtype = np.dtype(self.output_dtype)
        if len(df) == 0:
            return np.empty((0, len(self.feature_cols)), dtype=out_dtype)

        X = self._extract_float64(df)
        nan_mask = np.isnan(X)
        if np.any(nan_mask):
            X = np.where(nan_mask, self.medians_, X)

        X = np.clip(X, self.lower_bounds_, self.upper_bounds_)
        X = (X - self.scaler_.mean_) / self.scaler_.scale_
        X_out = X.astype(out_dtype, copy=False)
        if not np.all(np.isfinite(X_out)):
            raise ValueError("Non-finite values detected after FlowFeatureTransformer.transform().")
        return X_out

    def fit_transform(self, df_train: pd.DataFrame) -> np.ndarray:
        return self.fit(df_train).transform(df_train)

    def get_state(self) -> Dict[str, Any]:
        if not self.is_fitted_:
            raise RuntimeError("Cannot serialize an unfitted FlowFeatureTransformer.")
        return {
            "feature_cols": list(self.feature_cols),
            "clip_quantiles": [float(self.clip_quantiles[0]), float(self.clip_quantiles[1])],
            "output_dtype": self.output_dtype,
            "medians": [float(x) for x in self.medians_],
            "lower_bounds": [float(x) for x in self.lower_bounds_],
            "upper_bounds": [float(x) for x in self.upper_bounds_],
            "mean": [float(x) for x in self.scaler_.mean_],
            "scale": [float(x) for x in self.scaler_.scale_],
            "var": [float(x) for x in self.scaler_.var_],
            "n_samples_seen": int(self.scaler_.n_samples_seen_),
        }

    @classmethod
    def from_state(cls, state: Dict[str, Any]) -> "FlowFeatureTransformer":
        obj = cls(
            feature_cols=state["feature_cols"],
            clip_quantiles=(float(state["clip_quantiles"][0]), float(state["clip_quantiles"][1])),
            output_dtype=str(state.get("output_dtype", "float32")),
        )
        obj.medians_ = np.asarray(state["medians"], dtype=np.float64)
        obj.lower_bounds_ = np.asarray(state["lower_bounds"], dtype=np.float64)
        obj.upper_bounds_ = np.asarray(state["upper_bounds"], dtype=np.float64)

        scaler = StandardScaler()
        scaler.mean_ = np.asarray(state["mean"], dtype=np.float64)
        scaler.scale_ = np.asarray(state["scale"], dtype=np.float64)
        scaler.scale_ = np.where(scaler.scale_ < 1e-12, 1.0, scaler.scale_)
        scaler.var_ = np.asarray(state["var"], dtype=np.float64)
        scaler.n_features_in_ = len(obj.feature_cols)
        scaler.n_samples_seen_ = int(state.get("n_samples_seen", 0))
        obj.scaler_ = scaler
        obj.is_fitted_ = True
        return obj


class TargetLabelEncoder:
    """Fits known `attack_family` classes on Train; maps unseen or missing labels safely to -1."""

    def __init__(self, unknown_index: int = -1, unknown_label: str = "UNKNOWN_UNSEEN") -> None:
        self.unknown_index: int = int(unknown_index)
        self.unknown_label: str = str(unknown_label)
        self.classes_: List[str] = []
        self.class_to_idx_: Dict[str, int] = {}
        self.idx_to_class_: Dict[int, str] = {}
        self.is_fitted_: bool = False

    def fit(self, y: Any) -> "TargetLabelEncoder":
        values = list(y)
        if len(values) == 0:
            raise ValueError("Cannot fit TargetLabelEncoder on empty labels.")
        unique_classes = sorted({str(v) for v in values if pd.notna(v)})
        if not unique_classes:
            raise ValueError("TargetLabelEncoder found no non-null label classes to fit.")
        self.classes_ = unique_classes
        self.class_to_idx_ = {c: i for i, c in enumerate(self.classes_)}
        self.idx_to_class_ = {i: c for i, c in enumerate(self.classes_)}
        self.is_fitted_ = True
        return self

    def transform(self, y: Any) -> np.ndarray:
        if not self.is_fitted_:
            raise RuntimeError("TargetLabelEncoder must be fitted before calling transform().")
        encoded = [
            self.class_to_idx_.get(str(v), self.unknown_index) if pd.notna(v) else self.unknown_index
            for v in y
        ]
        return np.asarray(encoded, dtype=np.int64)

    def fit_transform(self, y: Any) -> np.ndarray:
        return self.fit(y).transform(y)

    def inverse_transform(self, indices: Sequence[int]) -> List[str]:
        if not self.is_fitted_:
            raise RuntimeError("TargetLabelEncoder must be fitted before calling inverse_transform().")
        return [self.idx_to_class_.get(int(idx), self.unknown_label) for idx in indices]

    def known_mask(self, y_encoded: Sequence[int]) -> np.ndarray:
        return np.asarray(y_encoded, dtype=np.int64) != self.unknown_index

    def get_state(self) -> Dict[str, Any]:
        if not self.is_fitted_:
            raise RuntimeError("Cannot serialize an unfitted TargetLabelEncoder.")
        return {
            "classes": list(self.classes_),
            "unknown_index": self.unknown_index,
            "unknown_label": self.unknown_label,
        }

    @classmethod
    def from_state(cls, state: Dict[str, Any]) -> "TargetLabelEncoder":
        obj = cls(
            unknown_index=int(state.get("unknown_index", -1)),
            unknown_label=str(state.get("unknown_label", "UNKNOWN_UNSEEN")),
        )
        obj.classes_ = [str(c) for c in state["classes"]]
        obj.class_to_idx_ = {c: i for i, c in enumerate(obj.classes_)}
        obj.idx_to_class_ = {i: c for i, c in enumerate(obj.classes_)}
        obj.is_fitted_ = True
        return obj


transformer = FlowFeatureTransformer().fit(splits["train"])
label_encoder = TargetLabelEncoder().fit(splits["train"]["attack_family"])
print(f"Transformer fitted on Train ({len(splits['train']):,} rows, {len(L3_L4_FEATURE_COLUMNS)} features).")
print(f"Known Train attack_family classes ({len(label_encoder.classes_)}): {label_encoder.classes_[:10]}...")

Transformer fitted on Train (425,771 rows, 40 features).
Known Train attack_family classes (19): ['Analysis', 'Backdoor', 'Benign', 'Bot', 'Brute Force', 'DDoS', 'DoS', 'Exploits', 'Fuzzers', 'Generic']...


## Cell 4 — AnomalyFlowDataset Abstraction

In [4]:
class AnomalyFlowDataset:
    """Prepares benign-only Train baseline, Validation calibration set, Test, and paired Unseen Holdout."""

    def __init__(self, splits: Dict[str, pd.DataFrame], transformer: FlowFeatureTransformer) -> None:
        self.df_train = splits["train"].reset_index(drop=True)
        self.df_val = splits["val"].reset_index(drop=True)
        self.df_test = splits["test"].reset_index(drop=True)
        self.df_unseen = splits["unseen_holdout"].reset_index(drop=True)

        self.X_train = transformer.transform(self.df_train)
        self.y_train = self.df_train["label"].to_numpy(dtype=np.int64)
        self.X_val = transformer.transform(self.df_val)
        self.y_val = self.df_val["label"].to_numpy(dtype=np.int64)
        self.X_test = transformer.transform(self.df_test)
        self.y_test = self.df_test["label"].to_numpy(dtype=np.int64)

        self.df_test_benign = self.df_test[self.df_test["label"] == 0].reset_index(drop=True)
        self.X_test_benign = transformer.transform(self.df_test_benign)
        self.y_test_benign = self.df_test_benign["label"].to_numpy(dtype=np.int64)
        self.X_unseen = transformer.transform(self.df_unseen)
        self.y_unseen = self.df_unseen["label"].to_numpy(dtype=np.int64)


anomaly_ds = AnomalyFlowDataset(splits, transformer)
print(f"Train benign rows: {(anomaly_ds.y_train == 0).sum():,} | Val rows: {len(anomaly_ds.y_val):,} | Unseen holdout rows: {len(anomaly_ds.y_unseen):,}")

Train benign rows: 104,351 | Val rows: 91,230 | Unseen holdout rows: 635


## Cell 5 — IsolationForestThreatModel (Benign Fit + SR-04 Calibration)

In [5]:
from sklearn.ensemble import IsolationForest


class IsolationForestThreatModel:
    """Benign-only Isolation Forest with Validation-set FPR threshold calibration (SR-04)."""

    def __init__(
        self,
        n_estimators: int = 200,
        max_samples: Any = "auto",
        contamination: Any = "auto",
        max_features: float = 1.0,
        random_state: int = 42,
        n_jobs: int = -1,
    ) -> None:
        self.model_name = "week6_isolation_forest"
        self.task_type = "anomaly"
        self.n_estimators = int(n_estimators)
        self.max_samples = max_samples
        self.contamination = contamination
        self.max_features = float(max_features)
        self.random_state = int(random_state)
        self.n_jobs = int(n_jobs)
        self.estimator = IsolationForest(
            n_estimators=self.n_estimators,
            max_samples=self.max_samples,
            contamination=self.contamination,
            max_features=self.max_features,
            random_state=self.random_state,
            n_jobs=self.n_jobs,
        )
        self.benign_only_fit: bool = True
        self.n_train_benign: int = 0
        self.calibrated_threshold_: Optional[float] = None
        self.target_fpr_: float = 0.05
        self.score_scale_: float = 1.0
        self.val_benign_score_mean_: float = 0.0
        self.val_benign_score_std_: float = 1.0
        self.is_fitted: bool = False

    @staticmethod
    def _to_numpy(X: Any) -> np.ndarray:
        arr = np.asarray(X.X if hasattr(X, "X") else X, dtype=np.float32)
        if not np.isfinite(arr).all():
            raise ValueError("Input features contain NaN or Inf. Run FlowFeatureTransformer first.")
        return arr

    def fit(self, X: Any, y: Optional[Any] = None) -> "IsolationForestThreatModel":
        X_arr = self._to_numpy(X)
        if y is not None:
            y_arr = np.asarray(y)
            benign_mask = y_arr == 0
            if not np.any(benign_mask):
                raise ValueError("IsolationForest requires at least one benign (label == 0) row.")
            X_benign = X_arr[benign_mask]
        else:
            X_benign = X_arr
        if len(X_benign) == 0:
            raise ValueError("Cannot fit IsolationForest on empty array.")

        self.estimator.fit(X_benign)
        self.n_train_benign = int(len(X_benign))
        self.benign_only_fit = True
        self.is_fitted = True

        scores = self.score_samples(X_benign)
        self.calibrated_threshold_ = float(np.quantile(scores, 0.95))
        std = float(np.std(scores))
        self.score_scale_ = std if std > 1e-6 else 1.0
        return self

    def score_samples(self, X: Any) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("IsolationForestThreatModel must be fitted before scoring.")
        return np.asarray(-self.estimator.decision_function(self._to_numpy(X)), dtype=np.float64)

    def calibrate_threshold(self, X_val: Any, y_val: np.ndarray, target_fpr: float = 0.05) -> Dict[str, Any]:
        if not self.is_fitted:
            raise RuntimeError("Model must be fitted before calibrate_threshold().")
        if not (0.0 < target_fpr < 1.0):
            raise ValueError(f"target_fpr must be in (0, 1), got {target_fpr}.")

        y_arr = np.asarray(y_val)
        scores = self.score_samples(X_val)
        benign_mask = y_arr == 0
        if not np.any(benign_mask):
            raise ValueError("Validation split must contain benign (label == 0) rows for FPR calibration.")

        benign_scores = scores[benign_mask]
        threshold = float(np.quantile(benign_scores, 1.0 - target_fpr))
        self.calibrated_threshold_ = threshold
        self.target_fpr_ = float(target_fpr)
        self.val_benign_score_mean_ = float(np.mean(benign_scores))
        std = float(np.std(benign_scores))
        self.val_benign_score_std_ = std if std > 1e-6 else 1.0
        self.score_scale_ = self.val_benign_score_std_

        preds = (scores >= threshold).astype(np.int64)
        return {
            "calibrated_threshold": threshold,
            "target_fpr": float(target_fpr),
            "achieved_val_fpr": float(preds[benign_mask].mean()),
            "achieved_val_tpr": float(preds[~benign_mask].mean()) if np.any(~benign_mask) else 0.0,
            "n_val_benign": int(benign_mask.sum()),
            "n_val_attack": int((~benign_mask).sum()),
        }

    def predict(self, X: Any, threshold: Optional[float] = None) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("Model must be fitted before predict().")
        thr = float(threshold) if threshold is not None else float(self.calibrated_threshold_)
        return (self.score_samples(X) >= thr).astype(np.int64)

    def predict_with_context(
        self,
        X: Any,
        df_raw: Optional[pd.DataFrame] = None,
        threshold: Optional[float] = None,
    ) -> Dict[str, Any]:
        if not self.is_fitted:
            raise RuntimeError("Model must be fitted before predict_with_context().")
        thr = float(threshold) if threshold is not None else float(self.calibrated_threshold_)
        scores = self.score_samples(X)
        preds = (scores >= thr).astype(np.int64)
        z = np.clip((scores - thr) / max(self.score_scale_, 1e-6), -30.0, 30.0)
        confidence = (1.0 / (1.0 + np.exp(-z))).astype(np.float64)

        n = len(scores)
        n_raw = len(df_raw) if df_raw is not None else 0
        if df_raw is not None and n_raw > 0:
            tls_arr: Optional[np.ndarray] = (
                pd.to_numeric(df_raw["is_tls_port"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
                if "is_tls_port" in df_raw.columns
                else None
            )
            ssh_arr: Optional[np.ndarray] = (
                pd.to_numeric(df_raw["is_ssh_port"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
                if "is_ssh_port" in df_raw.columns
                else None
            )
            ratio_arr: Optional[np.ndarray] = (
                pd.to_numeric(df_raw["bytes_ratio"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
                if "bytes_ratio" in df_raw.columns
                else None
            )
        else:
            tls_arr = ssh_arr = ratio_arr = None

        reasons: List[List[str]] = []
        for i in range(n):
            row_reasons: List[str] = []
            if preds[i] == 1:
                row_reasons.append(f"ANOMALY_SCORE_ABOVE_CALIBRATED_FPR_{self.target_fpr_:.3f}")
                if scores[i] >= thr + 2.0 * self.score_scale_:
                    row_reasons.append("EXTREME_ISOLATION_DEPTH_OUTLIER")
                if df_raw is not None and i < n_raw:
                    if tls_arr is not None and tls_arr[i] == 1.0:
                        row_reasons.append("CONTEXT_TLS_PORT_FLOW")
                    elif ssh_arr is not None and ssh_arr[i] == 1.0:
                        row_reasons.append("CONTEXT_SSH_PORT_FLOW")
                    if ratio_arr is not None and ratio_arr[i] > 50.0:
                        row_reasons.append("CONTEXT_HIGH_BYTES_ASYMMETRY")
            else:
                row_reasons.append("WITHIN_BENIGN_CALIBRATED_BASELINE")
            reasons.append(row_reasons)

        return {
            "predictions": preds,
            "anomaly_scores": scores,
            "confidence": confidence,
            "reason_codes": reasons,
            "calibrated_threshold": thr,
            "target_fpr": self.target_fpr_,
        }

    def get_metadata(self) -> Dict[str, Any]:
        return {
            "model_name": self.model_name,
            "task_type": self.task_type,
            "n_estimators": self.n_estimators,
            "max_samples": str(self.max_samples),
            "contamination": str(self.contamination),
            "max_features": self.max_features,
            "random_state": self.random_state,
            "benign_only_fit": self.benign_only_fit,
            "n_train_benign": self.n_train_benign,
            "calibrated_threshold": self.calibrated_threshold_,
            "target_fpr": self.target_fpr_,
            "score_scale": self.score_scale_,
            "val_benign_score_mean": self.val_benign_score_mean_,
            "val_benign_score_std": self.val_benign_score_std_,
            "is_fitted": self.is_fitted,
        }

    @classmethod
    def from_checkpoint(cls, payload: Any) -> "IsolationForestThreatModel":
        if isinstance(payload, cls):
            return payload
        if isinstance(payload, dict):
            if "model" in payload and isinstance(payload["model"], cls):
                return payload["model"]
            if "estimator" in payload:
                est = payload["estimator"]
                state = payload.get("model_state") or payload.get("state") or {}
                obj = cls(
                    n_estimators=int(state.get("n_estimators", getattr(est, "n_estimators", 200))),
                    max_samples=getattr(est, "max_samples", state.get("max_samples", "auto")),
                    contamination=getattr(est, "contamination", state.get("contamination", "auto")),
                    max_features=float(state.get("max_features", getattr(est, "max_features", 1.0))),
                    random_state=int(state.get("random_state", getattr(est, "random_state", 42) or 42)),
                    n_jobs=int(getattr(est, "n_jobs", state.get("n_jobs", -1)) or -1),
                )
                obj.estimator = est
                obj.benign_only_fit = bool(state.get("benign_only_fit", True))
                obj.n_train_benign = int(state.get("n_train_benign", 0))
                cal_thr = state.get("calibrated_threshold")
                obj.calibrated_threshold_ = float(cal_thr) if cal_thr is not None else None
                obj.target_fpr_ = float(state.get("target_fpr", 0.05))
                obj.score_scale_ = float(state.get("score_scale", 1.0))
                obj.val_benign_score_mean_ = float(state.get("val_benign_score_mean", 0.0))
                obj.val_benign_score_std_ = float(state.get("val_benign_score_std", 1.0))
                obj.is_fitted = bool(state.get("is_fitted", hasattr(est, "estimators_")))
                return obj
        raise TypeError(f"Unsupported IsolationForest checkpoint payload type: {type(payload).__name__}")

## Cell 6 — Trainer, Evaluator & CheckpointManager

In [6]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


class Week6Trainer:
    """Orchestrates benign-only fit and validation FPR threshold calibration."""

    def __init__(self, model: IsolationForestThreatModel, target_fpr: float = 0.05) -> None:
        self.model = model
        self.target_fpr = float(target_fpr)

    def train(self, ds: AnomalyFlowDataset) -> Dict[str, Any]:
        t0 = time.perf_counter()
        self.model.fit(ds.X_train, ds.y_train)
        fit_s = time.perf_counter() - t0
        calib = self.model.calibrate_threshold(ds.X_val, ds.y_val, target_fpr=self.target_fpr)
        return {"fit_seconds": fit_s, "n_train_benign": self.model.n_train_benign, "calibration": calib}


class Week6Evaluator:
    """Evaluates AUCPR, ROC-AUC, precision/recall, latency, and detection rate at fixed FPR."""

    @staticmethod
    def evaluate_pair(model: IsolationForestThreatModel, X: np.ndarray, y: np.ndarray) -> Dict[str, Any]:
        t0 = time.perf_counter()
        scores = model.score_samples(X)
        preds = (scores >= float(model.calibrated_threshold_)).astype(np.int64)
        elapsed = time.perf_counter() - t0

        yt = np.asarray(y, dtype=np.int64)
        cm = confusion_matrix(yt, preds, labels=[0, 1])
        tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
        aucpr = float(average_precision_score(yt, scores)) if len(np.unique(yt)) > 1 else float(np.mean(yt == 1))
        roc = float(roc_auc_score(yt, scores)) if len(np.unique(yt)) > 1 else 0.5
        achieved_tpr = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
        tpr_fixed: Dict[str, float] = {}
        if np.any(yt == 0) and np.any(yt == 1):
            for fpr_t in (0.01, 0.05, 0.10):
                q = float(np.quantile(scores[yt == 0], 1.0 - fpr_t))
                tpr_fixed[f"tpr_at_fpr_{fpr_t:.2f}"] = float((scores[yt == 1] >= q).mean())
        else:
            for fpr_t in (0.01, 0.05, 0.10):
                tpr_fixed[f"tpr_at_fpr_{fpr_t:.2f}"] = achieved_tpr
        return {
            "calibrated_threshold": float(model.calibrated_threshold_),
            "target_fpr": float(model.target_fpr_),
            "aucpr": aucpr,
            "roc_auc": roc,
            "f1": float(f1_score(yt, preds, zero_division=0)),
            "precision": float(precision_score(yt, preds, zero_division=0)),
            "recall": float(recall_score(yt, preds, zero_division=0)),
            "achieved_fpr": float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
            "tpr_at_fpr": achieved_tpr,
            "tpr_at_fixed_fpr": tpr_fixed,
            "confusion_matrix": cm.tolist(),
            "n_samples": int(len(yt)),
            "inference_latency_ms_per_flow": float((elapsed / max(1, len(yt))) * 1000.0),
        }

    def evaluate_unseen_holdout(
        self,
        model: IsolationForestThreatModel,
        X_unseen: np.ndarray,
        y_unseen: np.ndarray,
        X_benign_test: np.ndarray,
        y_benign_test: np.ndarray,
    ) -> Dict[str, Any]:
        X_b = np.asarray(X_benign_test, dtype=np.float32)
        y_b = np.asarray(y_benign_test, dtype=np.int64)
        X_u = np.asarray(X_unseen, dtype=np.float32)
        y_u = np.asarray(y_unseen, dtype=np.int64)
        if len(X_b) == 0 or len(y_b) == 0 or not np.any(y_b == 0):
            raise ValueError("evaluate_unseen_holdout() requires non-empty benign (label == 0) test rows.")
        if len(X_u) == 0 or len(y_u) == 0:
            raise ValueError("evaluate_unseen_holdout() requires non-empty unseen attack rows.")
        metrics = self.evaluate_pair(model, np.vstack([X_b, X_u]), np.concatenate([y_b, y_u]))
        metrics["n_benign_paired"] = int(len(y_b))
        metrics["n_unseen_attacks"] = int(len(y_u))
        return metrics


class CheckpointManager:
    """Saves/reloads `.joblib` artifacts + sanitized `_metadata.json` provenance (`allow_nan=False`)."""

    def __init__(self, ckpt_dir: Path) -> None:
        self.ckpt_dir = Path(ckpt_dir)
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)

    @classmethod
    def _sanitize_json(cls, obj: Any) -> Any:
        if obj is None or isinstance(obj, (bool, str, int)):
            return obj
        if isinstance(obj, float):
            return obj if math.isfinite(obj) else None
        if isinstance(obj, Path):
            return str(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            val = float(obj)
            return val if math.isfinite(val) else None
        if isinstance(obj, np.ndarray):
            return cls._sanitize_json(obj.tolist())
        if isinstance(obj, Mapping):
            return {str(k): cls._sanitize_json(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple, set, frozenset)):
            return [cls._sanitize_json(v) for v in obj]
        return str(obj)

    def save(self, name: str, payload: Dict[str, Any], metadata: Dict[str, Any]) -> Tuple[Path, Path]:
        art_path = self.ckpt_dir / f"{name}.joblib"
        meta_path = self.ckpt_dir / f"{name}_metadata.json"
        joblib.dump(payload, art_path, compress=3)
        clean_meta = self._sanitize_json(metadata)
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(clean_meta, f, indent=2, ensure_ascii=False, allow_nan=False)
        return art_path, meta_path

    def load(self, name: str) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        payload = joblib.load(self.ckpt_dir / f"{name}.joblib")
        with open(self.ckpt_dir / f"{name}_metadata.json", "r", encoding="utf-8") as f:
            meta = json.load(f)
        return payload, meta

## Cell 7 — Training, SR-04 Calibration, Unseen Attack Evaluation & Checkpointing

In [7]:
n_trees = 50 if PATHS["is_sample_mode"] else 200
if_model = IsolationForestThreatModel(n_estimators=n_trees, random_state=CONFIG.seed)
train_res = Week6Trainer(if_model, target_fpr=0.05).train(anomaly_ds)
print("=== TRAIN & VALIDATION CALIBRATION (SR-04) ===")
print(json.dumps(train_res, indent=2))

evaluator = Week6Evaluator()
test_metrics = evaluator.evaluate_pair(if_model, anomaly_ds.X_test, anomaly_ds.y_test)

# Paired Unseen Attack Holdout evaluation (Unseen attacks + Benign test flows)
unseen_metrics = evaluator.evaluate_unseen_holdout(
    model=if_model,
    X_unseen=anomaly_ds.X_unseen,
    y_unseen=anomaly_ds.y_unseen,
    X_benign_test=anomaly_ds.X_test_benign,
    y_benign_test=anomaly_ds.y_test_benign,
)

ctx_sample = if_model.predict_with_context(anomaly_ds.X_unseen[:5], df_raw=anomaly_ds.df_unseen.head(5))
print("\n=== FULL TEST METRICS ===")
print(json.dumps(test_metrics, indent=2))
print("\n=== UNSEEN ATTACK HOLDOUT METRICS (PAIRED WITH BENIGN TEST) ===")
print(json.dumps(unseen_metrics, indent=2))
print("\nSample SR-04 Reason Codes:", ctx_sample["reason_codes"][:3])

ckpt_mgr = CheckpointManager(PATHS["checkpoint_dir"])
art_path, meta_path = ckpt_mgr.save(
    "week6_isolation_forest",
    {
        "estimator": if_model.estimator,
        "model_state": if_model.get_metadata(),
        "transformer_state": transformer.get_state(),
    },
    {
        "seed": CONFIG.seed,
        "versions": VERSIONS,
        "environment": VERSIONS,
        "feature_columns": L3_L4_FEATURE_COLUMNS,
        "calibrated_threshold": if_model.calibrated_threshold_,
        "target_fpr": if_model.target_fpr_,
        "test_metrics": test_metrics,
        "unseen_holdout_metrics": unseen_metrics,
        "saved_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    },
)

reloaded_payload, _ = ckpt_mgr.load("week6_isolation_forest")
reloaded_if = IsolationForestThreatModel.from_checkpoint(reloaded_payload)
reloaded_tf = FlowFeatureTransformer.from_state(reloaded_payload["transformer_state"])
assert np.allclose(
    if_model.score_samples(anomaly_ds.X_test),
    reloaded_if.score_samples(reloaded_tf.transform(anomaly_ds.df_test)),
    atol=1e-6,
)
print(f"\n[VERIFIED] Saved & reloaded cross-env checkpoint: {art_path.name}")

=== TRAIN & VALIDATION CALIBRATION (SR-04) ===
{
  "fit_seconds": 1.3511294620000172,
  "n_train_benign": 104351,
  "calibration": {
    "calibrated_threshold": 0.0593456245104671,
    "target_fpr": 0.05,
    "achieved_val_fpr": 0.05008720540226287,
    "achieved_val_tpr": 0.12833059867284263,
    "n_val_benign": 22361,
    "n_val_attack": 68869
  }
}

=== FULL TEST METRICS ===
{
  "calibrated_threshold": 0.0593456245104671,
  "target_fpr": 0.05,
  "aucpr": 0.9123636812049641,
  "roc_auc": 0.8549242609312248,
  "f1": 0.212609487468563,
  "precision": 0.9086730911786508,
  "recall": 0.12038892162639953,
  "achieved_fpr": 0.037010334054313865,
  "tpr_at_fpr": 0.12038892162639953,
  "tpr_at_fixed_fpr": {
    "tpr_at_fpr_0.01": 0.010989982321744255,
    "tpr_at_fpr_0.05": 0.17051659791789434,
    "tpr_at_fpr_0.10": 0.40114908662345317
  },
  "confusion_matrix": [
    [
      32056,
      1232
    ],
    [
      89562,
      12258
    ]
  ],
  "n_samples": 135108,
  "inference_latency_ms_pe